# Competitive shootout: mboostrs against libraries on thirteen datasets

This study asks how far mboostrs' models are from the best widely used libraries on
the same data. It uses thirteen datasets: ten real datasets and three generated panels,
each with a known structure. Each library was given the same train and test splits.
The study froze its workflows before scoring, so the comparison measures the methods
as they were set up, not as they were later improved.

The measure is excess loss. For each dataset, the best competitor's test loss is the
reference, and a method's excess is how much worse its loss is, as a percentage of
that reference. Zero means the method matched the best competitor. A high excess means
the method was well behind it.

The report is `docs/benchmarks/competitive_shootout_report.md`. Its per-split records
are in `competitive_shootout_scores.csv`, and the per-cell summary used here is in
`competitive_shootout_summary.json`.

## Methods and tiers

The study compares eight methods: three mboostrs variants and five competitors:

- **mboostrs additive**: one smooth curve per input, with no interactions.
- **mboostrs hybrid**: a smooth curve per input, plus one tree that may use several inputs.
- **mboostrs autospec**: the library's search over model forms, with a fixed budget.
- **Linear**: a linear or logistic model, used as a floor.
- **EBM**, **CatBoost**, **LightGBM** and **XGBoost**: the competitor libraries.

Each method runs at two tiers. The **small** tier uses default settings. The **tuned**
tier searches the settings with a fixed budget. This page uses the tuned tier, which is
the one the report uses for its accuracy table.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the summary sits in the benchmarks directory.
SUMMARY = Path("evidence/competitive_shootout/competitive_shootout_summary.json")
summary = json.loads(SUMMARY.read_text())

cells = pl.DataFrame(
    [
        {
            "task": c["task"],
            "method": c["method"],
            "tier": c["tier"],
            "complete": bool(c["complete"]),
            "excess_pct": c["excess_pct"],
        }
        for c in summary["cells"]
    ],
    schema_overrides={"excess_pct": pl.Float64},
)
print(f"cells: {cells.height}; methods: {cells['method'].n_unique()}; tasks: {cells['task'].n_unique()}")
print("failed workflows recorded in the summary:", summary["failure_count"])

cells: 205; methods: 8; tasks: 13
failed workflows recorded in the summary: 12


## Headline by method

The table uses the ten real datasets, at the tuned tier. For each method it gives the
median excess loss across those datasets, the number of datasets where the method is
within 5% of the best competitor, and the median time per workflow from the summary.
Bold marks the lowest median excess.

In [2]:
REAL = ["housing_iid", "housing_spatial", "bike_iid", "bike_future", "diamonds", "protein",
        "superconductivity", "concrete", "credit", "cancer"]
tuned_real = cells.filter((pl.col("tier") == "tuned") & pl.col("task").is_in(REAL) & pl.col("complete"))

headline = (
    tuned_real.group_by("method")
    .agg(
        pl.col("excess_pct").median().alias("median"),
        (pl.col("excess_pct") <= 5).sum().alias("within5"),
        pl.len().alias("datasets"),
    )
    .sort("median")
)
times = {h["method"]: h["median_workflow_s"] for h in summary["headline"]}

best_median = headline["median"].min()
rows = []
for r in headline.iter_rows(named=True):
    median_text = f"{r['median']:.1f}%"
    rows.append(
        {
            "Method": r["method"],
            "Median excess": f"**{median_text}**" if r["median"] == best_median else median_text,
            "Within 5% of best": f"{r['within5']} of {r['datasets']}",
            "Median seconds per workflow": f"{times.get(r['method'], float('nan')):.2f}",
        }
    )

headline_table = pl.DataFrame(rows)
(
    GT(headline_table)
    .tab_header(
        title="Excess test loss by method on ten real datasets",
        subtitle="Tuned tier; excess over the best competitor on each dataset; lower is better",
    )
    .fmt_markdown(columns=["Median excess"])
    .tab_stubhead(label="Method")
    .tab_source_note(
        "Excess: percent by which the method's test loss exceeds the best competitor's on the same dataset. "
        "Seconds: median wall-clock time per workflow in the summary. Source: competitive_shootout_summary.json."
    )
)

GT(_tbl_data=shape: (8, 4)
┌─────────────────┬───────────────┬───────────────────┬─────────────────────────────┐
│ Method          ┆ Median excess ┆ Within 5% of best ┆ Median seconds per workflow │
│ ---             ┆ ---           ┆ ---               ┆ ---                         │
│ str             ┆ str           ┆ str               ┆ str                         │
╞═════════════════╪═══════════════╪═══════════════════╪═════════════════════════════╡
│ catboost        ┆ **0.1%**      ┆ 8 of 10           ┆ nan                         │
│ ebm             ┆ 5.5%          ┆ 4 of 9            ┆ nan                         │
│ lightgbm        ┆ 6.8%          ┆ 3 of 10           ┆ nan                         │
│ xgboost         ┆ 7.9%          ┆ 3 of 10           ┆ nan                         │
│ mboost_hybrid   ┆ 8.4%          ┆ 2 of 10           ┆ nan                         │
│ mboost_auto     ┆ 38.3%         ┆ 2 of 9            ┆ nan                         │
│ mboost_additive ┆ 46.5%         ┆ 1 of 10           ┆ nan                         │
│ linear          ┆ 132.3%        ┆ 2 of 10           ┆ nan                         │
└─────────────────┴───────────────┴───────────────────┴─────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10a768aa0>, _boxhead=Boxhead([ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Median excess', type=<ColInfoTypeEnum.default: 1>, column_label='Median excess', column_align='left', column_width=None), ColInfo(var='Within 5% of best', type=<ColInfoTypeEnum.default: 1>, column_label='Within 5% of best', column_align='left', column_width=None), ColInfo(var='Median seconds per workflow', type=<ColInfoTypeEnum.default: 1>, column_label='Median seconds per workflow', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a104aa0>, _spanners=Spanners([]), _heading=Heading(title='Excess test loss by method on ten real datasets', subtitle='Tuned tier; excess over the best competitor on each dataset; lower is better', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10ae30290>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10b83e120>, _source_notes=["Excess: percent by which the method's test loss exceeds the best competitor's on the same dataset. Seconds: median wall-clock time per workflow in the summary. Source: competitive_shootout_summary.json."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10a547ce0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10a544c80>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', ty

CatBoost has the lowest median excess, 0.1%, and it is within 5% of the best competitor
on 8 of the 10 real datasets. The mboostrs hybrid has a median of 8.4% and is within 5%
on 2 datasets. The additive model has a median of 46.5%, and the search variant of
mboostrs has a median of 38.3%. The linear floor has a median of 132.3%.

## Accuracy on every dataset

The next table gives the excess for every dataset and method at the tuned tier, across
all thirteen datasets. The three generated panels are included. A cell reads
"incomplete" when at least one split failed or timed out, and the value is then not
inferred.

In [3]:
order = ["linear", "mboost_additive", "mboost_hybrid", "mboost_auto", "ebm", "catboost", "lightgbm", "xgboost"]
labels = {
    "linear": "Linear",
    "mboost_additive": "mboostrs additive",
    "mboost_hybrid": "mboostrs hybrid",
    "mboost_auto": "mboostrs autospec",
    "ebm": "EBM",
    "catboost": "CatBoost",
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
}
tasks = sorted(cells["task"].unique().to_list())

accuracy_rows = []
for task in tasks:
    entry = {"Dataset": task.replace("_", " ")}
    for method in order:
        match = cells.filter((pl.col("task") == task) & (pl.col("method") == method) & (pl.col("tier") == "tuned"))
        if match.height == 0:
            entry[labels[method]] = "—"
            continue
        row = match.row(0, named=True)
        if not row["complete"] or row["excess_pct"] is None:
            entry[labels[method]] = "incomplete"
        elif row["excess_pct"] < 0.5:
            entry[labels[method]] = "**0%**"
        else:
            entry[labels[method]] = f"{row['excess_pct']:+.0f}%"
    accuracy_rows.append(entry)

accuracy_table = pl.DataFrame(accuracy_rows)
(
    GT(accuracy_table)
    .tab_header(
        title="Excess test loss by dataset and method, tuned tier",
        subtitle="Percent over the best competitor on each dataset; 0% marks the best; lower is better",
    )
    .fmt_markdown(columns=[labels[m] for m in order])
    .tab_spanner(label="Competitor libraries", columns=["EBM", "CatBoost", "LightGBM", "XGBoost"])
    .tab_spanner(label="mboostrs", columns=["mboostrs additive", "mboostrs hybrid", "mboostrs autospec"])
    .tab_stubhead(label="Dataset")
    .tab_source_note(
        "Bold 0%: the method matched the best value on that dataset. 'incomplete': a split failed or timed out. "
        "'—': the method did not run on that dataset. Source: competitive_shootout_summary.json, tuned tier."
    )
)

GT(_tbl_data=shape: (13, 9)
┌─────────────┬────────┬─────────────┬────────────┬───┬────────────┬──────────┬──────────┬─────────┐
│ Dataset     ┆ Linear ┆ mboostrs    ┆ mboostrs   ┆ … ┆ EBM        ┆ CatBoost ┆ LightGBM ┆ XGBoost │
│ ---         ┆ ---    ┆ additive    ┆ hybrid     ┆   ┆ ---        ┆ ---      ┆ ---      ┆ ---     │
│ str         ┆ str    ┆ ---         ┆ ---        ┆   ┆ str        ┆ str      ┆ str      ┆ str     │
│             ┆        ┆ str         ┆ str        ┆   ┆            ┆          ┆          ┆         │
╞═════════════╪════════╪═════════════╪════════════╪═══╪════════════╪══════════╪══════════╪═════════╡
│ bike future ┆ +533%  ┆ +158%       ┆ +24%       ┆ … ┆ **0%**     ┆ +17%     ┆ +26%     ┆ +25%    │
│ bike iid    ┆ +982%  ┆ +269%       ┆ +11%       ┆ … ┆ +5%        ┆ **0%**   ┆ +4%      ┆ +9%     │
│ cancer      ┆ **0%** ┆ +7%         ┆ +109%      ┆ … ┆ +93%       ┆ +31%     ┆ +69%     ┆ +34%    │
│ concrete    ┆ +408%  ┆ +71%        ┆ +15%       ┆ … ┆ +6%        ┆ +2%      ┆ **0%**   ┆ +9%     │
│ correlated  ┆ +180%  ┆ **0%**      ┆ +3%        ┆ … ┆ **0%**     ┆ +2%      ┆ +4%      ┆ +3%     │
│ proxies     ┆        ┆             ┆            ┆   ┆            ┆          ┆          ┆         │
│ …           ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …        ┆ …        ┆ …       │
│ housing     ┆ +43%   ┆ +37%        ┆ +5%        ┆ … ┆ +5%        ┆ **0%**   ┆ +7%      ┆ +10%    │
│ spatial     ┆        ┆             ┆            ┆   ┆            ┆          ┆          ┆         │
│ joint       ┆ +2153% ┆ +2153%      ┆ +67%       ┆ … ┆ incomplete ┆ **0%**   ┆ +23%     ┆ +71%    │
│ boundary    ┆        ┆             ┆            ┆   ┆            ┆          ┆          ┆         │
│ protein     ┆ +54%   ┆ +43%        ┆ +9%        ┆ … ┆ +15%       ┆ **0%**   ┆ +8%      ┆ +6%     │
│ smooth      ┆ +177%  ┆ **0%**      ┆ +5%        ┆ … ┆ **0%**     ┆ +2%      ┆ +7%      ┆ +6%     │
│ sparse      ┆        ┆             ┆            ┆   ┆            ┆          ┆          ┆         │
│ superconduc ┆ +141%  ┆ +90%        ┆ +6%        ┆ … ┆ —          ┆ **0%**   ┆ **0%**   ┆ +1%     │
│ tivity      ┆        ┆             ┆            ┆   ┆            ┆          ┆          ┆         │
└─────────────┴────────┴─────────────┴────────────┴───┴────────────┴──────────┴──────────┴─────────┘, _body=<great_tables._gt_data.Body object at 0x10b8a7170>, _boxhead=Boxhead([ColInfo(var='Dataset', type=<ColInfoTypeEnum.default: 1>, column_label='Dataset', column_align='left', column_width=None), ColInfo(var='Linear', type=<ColInfoTypeEnum.default: 1>, column_label='Linear', column_align='left', column_width=None), ColInfo(var='mboostrs additive', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs additive', column_align='left', column_width=None), ColInfo(var='mboostrs hybrid', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs hybrid', column_align='left', column_width=None), ColInfo(var='mboostrs autospec', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs autospec', column_align='left', column_width=None), ColInfo(var='EBM', type=<ColInfoTypeEnum.default: 1>, column_label='EBM', column_align='left', column_width=None), ColInfo(var='CatBoost', type=<ColInfoTypeEnum.default: 1>, column_label='CatBoost', column_align='left', column_width=None), ColInfo(var='LightGBM', type=<ColInfoTypeEnum.default: 1>, column_label='LightGBM', column_align='left', column_width=None), ColInfo(var='XGBoost', type=<ColInfoTypeEnum.default: 1>, column_label='XGBoost', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10adf6b10>, _spanners=Spanners([SpannerInfo(spanner_id='Competitor libraries', spanner_level=0, spanner_label='Competitor libraries', spanner_units=None, spanner_pattern=None, vars=['EBM', 'CatBoost', 'LightGBM', 'XGBoost'], built=None), SpannerInfo(spanner_id='mboostrs', spanner_level=0, spanner_label='mboostrs', spanner_units=None, spanner_pattern=None, vars=['mb

The table shows where each method falls short. On the joint boundary dataset (one of the thirteen datasets), the
mboostrs autospec search reaches a loss 314% above the best competitor, while its hybrid
is 67% above. The report traces this to the search itself: it never selects the joint
representation that the hybrid uses.

## Gaps the report prioritises

The report lists six gaps in order of priority. Each has an acceptance test, the measured condition that a later study must meet to close the gap:

1. **Joint effects in `effect_model`.** The joint-panel PEHE (the root mean squared error of the individual effect estimates) is 1.255, down from 1.548. The joint tree is less interpretable than the additive curves, and the report
   records this as a limit.
2. **Hybrid accuracy against CatBoost.** The hybrid's median excess is 8.4%, and the acceptance test is a median at or below 5% on real datasets, with no dataset above
   10%.
3. **Autospec search.** Autospec's median excess is 38.3%, and on the joint boundary
   dataset it misses the representation that the hybrid finds.
4. **Autospec wall-clock budget.** On the housing dataset autospec takes about 100
   seconds against 1.8 for the hybrid, and all six of its superconductivity workflows hit
   the 240-second limit.
5. **Reading the joint tree.** A `btree` term (a regression tree term) has no per-input curves, so its
   marginal-curve plots show flat lines for a model whose surface is correct.
6. **Study design.** The marginal-curve workload scored the additive model and ridge regression (a linear model with a size penalty on its coefficients) identically on the joint boundary dataset, because both predict flat marginals. A
   two-variable surface test separates them.

## Failures

The summary records 12 failed workflows. The report's failure table lists them with their
causes. A cell with an incomplete split shows no excess value in the table above.

## Findings

- **CatBoost is the most accurate method on these datasets.** Its median excess is 0.1%,
  and it is within 5% on 8 of the 10 real datasets. It is behind on cancer, where a
  linear score is already close, and on bike future, where EBM is best.
- **The mboostrs hybrid is about 8% behind the best competitor on real data.** It is the
  best mboostrs configuration in this study. The gap is largest on the higher-dimensional
  tabular datasets.
- **The additive model is far behind, and the joint tree closes most of that gap.** The hybrid adds interaction capacity through its joint tree; this comparison
  does not isolate all effects of tuning.
- **Autospec lags the hand-built hybrid.** It has a median excess of 38.3% against 8.4%,
  and on the housing dataset it takes about 100 seconds against 1.8 for the hybrid. It does not reach the capacity that the hybrid
  already has.

## Limits

- **Screening, not ranking.** The study covers thirteen datasets: three paired splits on
  each real dataset and five draws on each generated panel. The report states that the results describe these datasets and do not rank the
  libraries in general.
- **Tuned tier at a fixed budget.** The excess values depend on the search budget each
  method received.
- **Frozen workflows.** The data and workflows were fixed before scoring, so the results describe the methods as they were set up for the study.
- **Generated panels and real datasets are pooled in the accuracy table.** The headline
  uses the real datasets only.

## Reproduction

The workflows and their inputs are frozen in the repository's benchmark cache, and the
report gives the runner. The summary in `competitive_shootout_summary.json` is the input to
this page.